# Entraînement DPO de l'assistant RAG (Colab / Kaggle)

Ce notebook entraîne un adaptateur LoRA par DPO sur les paires de préférences produites **sur ta
machine** par `training/build_preferences.py` (avec Ollama). Il ne fait que l'entraînement : la
génération des paires et l'évaluation se font en local.

Prérequis :
1. Un GPU : *Exécution > Modifier le type d'exécution > GPU (T4)* sur Colab, ou *Accelerator > GPU* sur Kaggle.
2. Un token Hugging Face avec accès accepté à `meta-llama/Llama-3.1-8B-Instruct`
   (page du modèle > *Agree and access repository*).
3. Le fichier `training/data/preferences.jsonl` généré en local.

Durée indicative sur T4 : de l'ordre d'une heure pour ~300 paires (1 époque) — à vérifier sur ton premier run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code du projet et dépendances

In [ ]:
REPO = "https://github.com/rayanghomsi10/Assistant-RAG.git"
!git clone --depth 1 {REPO} assistant-rag
%cd assistant-rag
!pip install -q -r requirements-train.txt

## 2. Connexion à Hugging Face (modèle Llama 3.1 sous licence)

In [ ]:
from huggingface_hub import login
login()  # colle ton token (scope « read »)

## 3. Paires de préférences

Téléverse `preferences.jsonl` (Colab : bouton ci-dessous ; Kaggle : *Add data* puis adapter le chemin).

In [ ]:
import os, shutil
os.makedirs("training/data", exist_ok=True)
try:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.move(name, "training/data/preferences.jsonl")
except ImportError:
    print("Hors Colab : copie le fichier dans training/data/preferences.jsonl")
!wc -l training/data/preferences.jsonl

## 4. Entraînement

Hyperparamètres par défaut : 1 époque, lr 5e-6, beta 0.1, LoRA r=16, batch effectif 8,
séquences jusqu'à 3072 tokens. En cas de manque de mémoire, baisser `--max-length` (les paires
trop longues sont écartées, pas tronquées) ou `--lora-r`.

In [ ]:
!python training/train_dpo.py --data training/data/preferences.jsonl --output training/outputs/dpo-lora

À regarder dans `training_summary.json` : `eval_rewards/accuracies` (part des paires de validation
où le modèle préfère la bonne réponse) et `eval_rewards/margins`. Une exactitude proche de 1 dès
le début peut signaler des paires trop faciles (rejets triviaux) ; proche de 0,5, le modèle n'a
rien appris de séparable.

In [ ]:
import json
print(json.dumps(json.load(open("training/outputs/dpo-lora/training_summary.json")), indent=2))

## 5. Récupérer l'adaptateur

In [ ]:
!cd training/outputs && zip -r dpo-lora.zip dpo-lora
try:
    from google.colab import files
    files.download("training/outputs/dpo-lora.zip")
except ImportError:
    print("Hors Colab : télécharge training/outputs/dpo-lora.zip depuis le panneau de sortie")

## 6. Ensuite, en local

```bash
unzip dpo-lora.zip -d training/outputs/
python training/export_ollama.py --adapter training/outputs/dpo-lora
ollama create rag-assistant-dpo -f training/outputs/ollama/Modelfile
python evaluation/run_eval.py --run-name dpo_v1 --model rag-assistant-dpo
python evaluation/compare_runs.py baseline_v2 dpo_v1
```